<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/PDF_Signer_Verifier_Metadata_STEP_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔏 PDF Signing & Authenticity Validator with Metadata Extraction
This notebook allows you to:
- Upload a preprint PDF
- Extract metadata (email, ORCID, institution)
- Generate full SHA-256 hash and embed it in:
  - The last page (as a visible stamp)
  - A `.sha256` file for archive
  - PDF metadata (custom field)
- Re-validate authenticity by comparing hashes
- Version 1.1.0
- STEP 1 of 3

In [1]:
# 📦 Install PyMuPDF
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 86.8 MB/s eta 0:00:00


In [2]:
# 📤 Upload PDF
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f'Uploaded: {pdf_path}')

Saving preprints-154933-presentation.pdf to preprints-154933-presentation.pdf
Uploaded: preprints-154933-presentation.pdf


In [3]:
# 🔎 Extract metadata from first page
import fitz, re
doc = fitz.open(pdf_path)
first_page_text = doc[0].get_text()
email = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', first_page_text)
orcid = re.findall(r'\d{4}-\d{4}-\d{4}-\d{4}', first_page_text)
inst = re.findall(r'.*(University|Institute|Department|College).*', first_page_text, re.IGNORECASE)
email_guess = email[0] if email else ''
orcid_guess = orcid[0] if orcid else ''
inst_guess = inst[0] if inst else ''
doc.close()
print('Email:', email_guess)
print('ORCID:', orcid_guess)
print('Institution:', inst_guess)

Email: avalamontes@Kapodistrian.edu.gr
ORCID: 0009-0008-5616-7746
Institution: 


In [4]:
# ✏️ Author Data Input
from datetime import datetime
author = input('Author Name: ')
email = input(f'Email [{email_guess}]: ') or email_guess
orcid = input(f'ORCID [{orcid_guess}]: ') or orcid_guess
institution = input(f'Institution [{inst_guess}]: ') or inst_guess
timestamp = datetime.utcnow().isoformat() + 'Z'

Author Name: Antonios Valamontes
Email [avalamontes@Kapodistrian.edu.gr]: 
ORCID [0009-0008-5616-7746]: 
Institution []: Kapodistrian Academy of Science


In [5]:
# 🔐 Generate SHA-256 Hash
import hashlib
with open(pdf_path, 'rb') as f:
    pdf_bytes = f.read()
    sha256_hash = hashlib.sha256(pdf_bytes).hexdigest()
print('✅ SHA-256:', sha256_hash)

✅ SHA-256: 03be5a2edeb1d5f27f239c54457ba6a30a249c7d09f9cea142d5ece4275056b1


In [6]:
# 🖋️ Stamp Last Page + Embed Metadata
doc = fitz.open(pdf_path)
last = doc[-1]
stamp = f"""Signed by: {author}\nEmail: {email}\nORCID: {orcid}\nInstitution: {institution}\nDate: {timestamp}\nSHA-256: {sha256_hash}"""
last.insert_textbox(fitz.Rect(50, last.rect.height - 130, 550, last.rect.height - 20), stamp, fontsize=8)
doc.set_metadata({
  "title": "Signed Preprint",
  "subject": f"SHA-256:{sha256_hash}",
  "keywords": f"signature, integrity, sha256:{sha256_hash}",
  "author": author
})
signed_file = 'signed_' + pdf_path
doc.save(signed_file)
doc.close()
print(f'Saved: {signed_file}')

Saved: signed_preprints-154933-presentation.pdf


In [7]:
# 📁 Save .sha256 file
sha_file = signed_file + '.sha256'
with open(sha_file, 'w') as f:
    f.write(f"{sha256_hash}  {signed_file}\n")
print(f'SHA256 saved: {sha_file}')

SHA256 saved: signed_preprints-154933-presentation.pdf.sha256


In [8]:
# ⬇️ Download signed PDF + hash
files.download(signed_file)
files.download(sha_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🔍 Validate PDF Authenticity

In [ ]:
# 🆚 Upload PDF to validate
uploaded = files.upload()
verify_pdf = list(uploaded.keys())[0]

In [ ]:
# 🧾 Upload hash or paste manually
try:
  uploaded = files.upload()
  sha_file = list(uploaded.keys())[0]
  with open(sha_file, 'r') as f:
    expected_hash = f.read().strip().split()[0]
except:
  expected_hash = input('Paste expected SHA-256 hash: ')

In [ ]:
# ✅ Validate SHA-256 hash
with open(verify_pdf, 'rb') as f:
    file_hash = hashlib.sha256(f.read()).hexdigest()
print('Expected:', expected_hash)
print('Computed:', file_hash)
if file_hash == expected_hash:
    print('✅ Authentic: The file has not been altered.')
else:
    print('❌ Not authentic: Hash mismatch.')